# 15 — DNA / RNA evaluation

Runs **in the same Colab session as notebooks 12 and 13**, after both model lines have been trained and
their members saved. It needs the variables those notebooks leave behind (`MEMBERS`, `CANDS`, `Y`, `PROT`,
`SPLIT_DF`, `FOLD_IDX`, `PRED_DIR`, `V3_ROOT`, `BASE`, `paired_bootstrap`, `best_mcc_threshold`).

What survived the analysis, and what did not:

| | |
|---|---|
| **kept** | DNA and RNA scored as two standalone tasks, the way metal / nuclear / small are scored |
| **dropped** | every attempt to fold the two specialists back into one nucleic-acid prediction |

The dropped line (routing by declared class, max, noisy-OR, percentile or isotonic calibration) was shown
to be an **oracle**: routing a protein to the DNA model is itself the statement "this protein binds DNA",
and 1470 of 1470 positives sit in 66 of 300 proteins. A control — adding a constant 0.2 to the nuclear
score of the 56 declared DNA proteins, using no specialist model at all — reaches 0.5348, beating the real
routed DNA model (0.5273). None of it is reported.

Order: **BL → FI → TASK → HIT → EX**.

## 1. BL — per-ligand comparison

bindEmbed21DL / bindNode24 against our models, per ligand class, on both test sets. Also loads the
DNA and RNA members notebooks 12 and 13 saved, and defines `KIND` (the BioLiP2 ligand class) and
`load_baseline`, which the later cells use.

In [ ]:
#%% BL COMPARE
# ============================================================
# BL. bindEmbed21DL / bindNode24 vs our models, per ligand class, on the two test sets.
#     Standalone: needs only what part 1 of the notebook already built (MEMBERS, CANDS, Y, PROT, SPLIT_DF,
#     PRED_DIR, paired_bootstrap) plus the member predictions notebooks 12 and 13 saved on Drive.
#
#     Those two published methods have ONE nucleic-acid output. Scoring them against the DNA target and against
#     the RNA target separately shows which class they are actually good at, and what our split buys.
#     Two views per cell, both reported:
#       AUPRC_all        every protein of the split (what the app faces: any protein may be uploaded)
#       AUPRC_within     only the proteins that bind that class (if users upload proteins they already know)
# ============================================================
import json
KIND = {k: {'D': 'dna', 'R': 'rna', 'B': 'both', 'U': 'unknown'}[v] for k, v in json.loads(r"""
{"A4FTK7":"D","B3LQW9":"U","B8FW11":"D","B8H4R9":"D","C1IFD2":"R","C3SZN7":"D","C3W947":"D","C8V0B5":"D","D0CAL0":"R","D0CBZ8":"R","D0CDQ7":"R","D5MNX7":"D","D6E269":"U","F0NDX5":"R","G2RUZ1":"D","L7IX95":"D","O15527":"D","O25841":"D","O28416":"B","O28417":"D","O29975":"D","O35013":"R","O43809":"R","O52512":"D","O59188":"D","O74859":"D","O87963":"D","O94408":"R","P00648":"D","P00651":"R","P00698":"R","P01127":"D","P02263":"D","P02281":"D","P02958":"D","P03004":"D","P03013":"D","P03018":"D","P03034":"D","P03036":"D","P03042":"D","P03050":"D","P03052":"D","P03206":"D","P03496":"R","P03536":"R","P03612":"R","P03697":"D","P03699":"D","P03882":"D","P04390":"D","P04392":"D","P04418":"D","P06134":"D","P06534":"D","P06897":"D","P06956":"D","P09132":"R","P09184":"D","P0A0I7":"D","P0A459":"D","P0A6C1":"D","P0A6X7":"D","P0A776":"R","P0A7D1":"R","P0A881":"D","P0A8J2":"D","P0A8U6":"D","P0A988":"D","P0ACH5":"D","P0ACP7":"D","P0ACT4":"D","P0AEK0":"D","P0AF28":"D","P0AFJ5":"D","P0AG51":"R","P0AGB6":"D","P0CI76":"D","P0CY08":"D","P10828":"D","P10970":"R","P10971":"R","P10972":"R","P11690":"R","P11746":"D","P12732":"R","P12733":"R","P12734":"R","P12737":"R","P13123":"D","P13393":"D","P14116":"R","P14119":"R","P14121":"R","P14135":"R","P16458":"D","P18138":"R","P19793":"D","P22450":"R","P22452":"R","P23657":"D","P23873":"D","P24321":"U","P32410":"R","P32411":"R","P32761":"D","P33905":"D","P34257":"D","P37108":"R","P39075":"D","P44558":"D","P44688":"D","P49458":"R","P50187":"D","P50465":"D","P56178":"D","P56930":"R","P60618":"R","P60619":"R","P60848":"R","P62661":"U","P62669":"R","P62799":"D","P62801":"D","P62805":"D","P63159":"D","P67809":"R","P68919":"R","P69202":"D","P69687":"R","P71650":"R","P76632":"D","P80377":"R","P80382":"U","P84131":"D","P84229":"D","P84233":"D","Q03243":"R","Q05783":"D","Q14493":"R","Q2G285":"D","Q32WH4":"D","Q37964":"D","Q45488":"D","Q46864":"D","Q46898":"B","Q4UNB3":"D","Q57468":"D","Q5FBS0":"D","Q5SHN3":"R","Q5SHP2":"R","Q5SJ65":"D","Q5SJ76":"R","Q5SJH3":"R","Q5SLP8":"R","Q5Y812":"D","Q64368":"R","Q6NS38":"D","Q74ML9":"B","Q7LFL8":"D","Q7MHK3":"D","Q8C6P8":"D","Q8DQG2":"D","Q8GGH0":"D","Q8IKH2":"D","Q8NMG3":"D","Q8QL27":"R","Q8SXK5":"D","Q8WQM6":"D","Q8YYB7":"U","Q8Z2A5":"D","Q95VR4":"D","Q97ZE3":"D","Q9BTM1":"D","Q9BTP7":"D","Q9DHS8":"D","Q9HJY3":"D","Q9HSF4":"D","Q9I0M3":"D","Q9KJ88":"D","Q9KXR9":"D","Q9LSX0":"D","Q9LW31":"D","Q9X7H4":"D"}
""").items()}   # BioLiP2 2026-09-29, resolution <= 3 A (test-set proteins; the full table also covers Dev)

TEST = ['TestSet300', 'TestSetNew46']
BASE_PRED = BASELINE_WORK/'predictions'
OFFB = {s: np.r_[0, np.cumsum(SPLIT_DF[s].sequence.str.len().to_numpy())] for s in TEST}

for t in ('dna', 'rna'):                                   # our two lines, from the members saved on Drive
    for tag in ('V1', 'MS'):
        for s_ in (0, 1, 2):
            n = f'{t}_{tag}_s{s_}'
            if n not in MEMBERS:
                z = np.load(PRED_DIR/f'{n}.npz', allow_pickle=False)
                MEMBERS[n] = {k: z[k] for k in ('oof', *SPLITS_EXT)}
        MEMBERS[f'{t}_{tag}_bag'] = {k: np.mean([MEMBERS[f'{t}_{tag}_s{i}'][k] for i in (0, 1, 2)], axis=0)
                                     for k in ('oof', *SPLITS_EXT)}
    MEMBERS[f'{t}_line'] = {k: 0.5 * MEMBERS[f'{t}_V1_bag'][k] + 0.5 * MEMBERS[f'{t}_MS_bag'][k]
                            for k in ('oof', *SPLITS_EXT)}
C3 = CANDS['C3_union']
MEMBERS.setdefault('nuclear_final', {k: sum(w * MEMBERS[m][k] for m, w in zip(C3['members'], C3['weights']) if w)
                                     for k in ('oof', *SPLITS_EXT)})


def load_baseline(method, split):
    fp = BASE_PRED/f'{method}__{split}__nuclear_binding.csv.gz'
    assert fp.exists(), f'missing {fp}'
    df = pd.read_csv(fp)
    df['prot_id'] = df.prot_id.astype(str)
    g = {p: sub.sort_values('residue_index').probability.to_numpy() for p, sub in df.groupby('prot_id')}
    out = []
    for r in SPLIT_DF[split].itertuples():
        a = g[str(r.prot_id)]
        assert len(a) == len(r.sequence), f'{method}/{split}/{r.prot_id}: {len(a)} != {len(r.sequence)}'
        out.append(a)
    return np.concatenate(out).astype(float)


P = {m: {s: load_baseline(m, s) for s in TEST} for m in ('bindEmbed21DL', 'bindNode24')}
P['ours_nuclear_model'] = {s: np.asarray(MEMBERS['nuclear_final'][s], float) for s in TEST}
P['ours_DNA_model'] = {s: np.asarray(MEMBERS['dna_line'][s], float) for s in TEST}
P['ours_RNA_model'] = {s: np.asarray(MEMBERS['rna_line'][s], float) for s in TEST}


def target_and_subset(split, t):
    ys, idx = [], []
    for i, r in enumerate(SPLIT_DF[split].itertuples()):
        nuc = np.asarray(getattr(r, LABEL), np.int8)
        k = KIND.get(str(r.prot_id), 'unknown') if nuc.sum() else 'negative'
        ys.append(nuc if k in (t, 'both') else np.zeros_like(nuc))
        if nuc.sum() and k in (t, 'both'):
            idx.append(np.arange(OFFB[split][i], OFFB[split][i + 1]))
    return np.concatenate(ys).astype(int), np.concatenate(idx)


rows, boots = [], []
for split in TEST:
    for t in ('dna', 'rna'):
        y, sub = target_and_subset(split, t)
        ours = f'ours_{t.upper()}_model'
        n_prot = sum(1 for r in SPLIT_DF[split].itertuples()
                     if np.asarray(getattr(r, LABEL)).sum() and KIND.get(str(r.prot_id)) in (t, 'both'))
        for m, d in P.items():
            rows.append({'split': split, 'target': t.upper(), 'method': m, 'n_proteins': n_prot,
                         'n_pos': int(y.sum()), 'AUPRC_all': float(average_precision_score(y, d[split])),
                         'AUPRC_within': float(average_precision_score(y[sub], d[split][sub]))})
        for base in ('bindEmbed21DL', 'bindNode24'):        # paired protein bootstrap, ours vs each baseline
            b = paired_bootstrap(y, P[base][split], P[ours][split], 0.5, 0.5, PROT[split])
            b = b[b.metric.eq('AUPRC')].copy()
            b.insert(0, 'comparison', f'{ours} vs {base}')
            b.insert(0, 'target', t.upper())
            b.insert(0, 'split', split)
            boots.append(b)

tab = pd.DataFrame(rows)
for split in TEST:
    for t in ('DNA', 'RNA'):
        v = tab[(tab.split == split) & (tab.target == t)].sort_values('AUPRC_all', ascending=False)
        print(f"\n===== {split} / {t} target — {int(v.n_proteins.iloc[0])} proteins, "
              f"{int(v.n_pos.iloc[0])} positive residues =====")
        print(v[['method', 'AUPRC_all', 'AUPRC_within']].round(4).to_string(index=False))

boot = pd.concat(boots, ignore_index=True)
print('\n===== paired protein bootstrap (B = 10000), AUPRC of our class model vs each baseline =====')
print(boot[['split', 'target', 'comparison', 'ref', 'new', 'delta', 'delta_CI95_lo', 'delta_CI95_hi',
            'P(new>ref)']].round(4).to_string(index=False))

OUT_DIR = BASE/'results'/'PER_LIGAND_VS_BASELINES'
OUT_DIR.mkdir(parents=True, exist_ok=True)
tab.to_csv(OUT_DIR/'per_ligand_table.csv', index=False)
boot.to_csv(OUT_DIR/'per_ligand_bootstrap_B10000.csv', index=False)
print('\nsaved ->', OUT_DIR)

## 2. FI — the three final models

`nuclear`, `dna`, `rna`: members, weights, frozen Dev thresholds, and their predictions on both test
sets. Everything below reads `FINALS` from here.

In [ ]:
#%% FI INFERENCE
# ============================================================
# FI. The three final models, and their inference on TestSet300 + TestSetNew46.
#
#   nuclear  C3_union (notebook 09)   0.2*V1_s0 + 0.4*V1+ESMIF+PROTT5_bag3 + 0.4*MS+ESMIF+PROTT5_bag3   35 ckpt
#            (MS_s0 is in the recipe with weight 0.0, so it is not loaded)
#   dna      S1_dna_pair (notebook 12)   0.50*dna_V1_bag + 0.50*dna_MS_bag                              30 ckpt
#   rna      S1_rna_pair (notebook 13)   0.65*rna_V1_bag + 0.35*rna_MS_bag                              30 ckpt
#            every *_bag = mean of 3 seeds, each seed = mean of its 5 official folds
#
# Thresholds are RECOMPUTED here from the Dev out-of-fold predictions of each model against its own target
# (MCC-optimal, the house convention), and asserted against the values the notebooks locked. For dna / rna the
# 6 Dev proteins BioLiP does not cover are excluded, exactly as during training. Needs the BL cell (KIND).
# ============================================================
assert 'KIND' in dir(), 'run the BL cell first (it defines KIND)'
C3 = CANDS['C3_union']
FINALS = {
    'nuclear': {'members': [m for m, w in zip(C3['members'], C3['weights']) if w],
                'weights': [w for w in C3['weights'] if w], 'expect_thr': 0.7078787878787879},
    'dna': {'members': ['dna_V1_bag', 'dna_MS_bag'], 'weights': [0.5, 0.5], 'expect_thr': 0.5643434343434344},
    'rna': {'members': ['rna_V1_bag', 'rna_MS_bag'], 'weights': [0.65, 0.35], 'expect_thr': 0.7771717171717172},
}
SPLITS_ALL = {'TestSet300': 'TestSet300', 'TestSetNew46': 'TestSetNew46'}


def dev_target(t):
    """Dev labels + keep-mask for one model's own target ('nuclear' uses the nuclear labels, no mask)."""
    ys, ms = [], []
    for r in SPLIT_DF['DevSet1010'].itertuples():
        nuc = np.asarray(getattr(r, LABEL), np.int8)
        if t == 'nuclear':
            ys.append(nuc); ms.append(np.ones_like(nuc))
            continue
        k = KIND.get(str(r.prot_id), 'unknown') if nuc.sum() else 'negative'
        ys.append(nuc if k in (t, 'both') else np.zeros_like(nuc))
        ms.append(np.zeros_like(nuc) if k == 'unknown' else np.ones_like(nuc))
    return np.concatenate(ys).astype(int), np.concatenate(ms).astype(bool)


def test_target(split, t):
    ys = []
    for r in SPLIT_DF[split].itertuples():
        nuc = np.asarray(getattr(r, LABEL), np.int8)
        if t == 'nuclear':
            ys.append(nuc)
            continue
        k = KIND.get(str(r.prot_id), 'unknown') if nuc.sum() else 'negative'
        ys.append(nuc if k in (t, 'both') else np.zeros_like(nuc))
    return np.concatenate(ys).astype(int)


for t, f in FINALS.items():
    f['oof'] = sum(w * np.asarray(MEMBERS[m]['oof'], float) for m, w in zip(f['members'], f['weights']))
    y, keep = dev_target(t)
    f['thr'] = float(best_mcc_threshold(y[keep], f['oof'][keep])[0])
    assert abs(f['thr'] - f['expect_thr']) < 1e-9, f"{t}: threshold {f['thr']} != locked {f['expect_thr']}"
    for s in SPLITS_ALL:
        f[s] = sum(w * np.asarray(MEMBERS[m][s], float) for m, w in zip(f['members'], f['weights']))
    print(f"{t:8s} = " + " + ".join(f'{w:g}*{m}' for m, w in zip(f['members'], f['weights']))
          + f"  | threshold {f['thr']:.4f} (locked value reproduced)")

OUT = BASE/'results'/'FINAL_INFERENCE_3_MODELS'
OUT.mkdir(parents=True, exist_ok=True)
rows = []
for t, f in FINALS.items():
    for split in SPLITS_ALL:
        df = SPLIT_DF[split]
        y = test_target(split, t)
        p = f[split]
        pred = (p >= f['thr']).astype(int)
        pid = np.concatenate([[str(r.prot_id)] * len(r.sequence) for r in df.itertuples()])
        ridx = np.concatenate([np.arange(len(r.sequence)) for r in df.itertuples()])
        aa = np.concatenate([list(r.sequence) for r in df.itertuples()])
        assert len(pid) == len(y) == len(p), (len(pid), len(y), len(p))
        pd.DataFrame({'prot_id': pid, 'residue_index': ridx + 1, 'aa': aa, 'y_true': y,
                      'probability': p, 'predicted': pred}).to_csv(
            OUT/f'{t}__{split}__residue_predictions.csv.gz', index=False)
        rows.append({'model': t, 'split': split, 'n_residues': len(y), 'n_true': int(y.sum()),
                     'n_predicted': int(pred.sum()), 'threshold': f['thr'],
                     'AUPRC': float(average_precision_score(y, p)), 'AUROC': float(roc_auc_score(y, p)),
                     'precision': float(precision_score(y, pred, zero_division=0)),
                     'recall': float(recall_score(y, pred, zero_division=0)),
                     'F1': float(f1_score(y, pred, zero_division=0)),
                     'MCC': float(matthews_corrcoef(y, pred))})
summary = pd.DataFrame(rows)
display(summary.round(4))
summary.to_csv(OUT/'SUMMARY.csv', index=False)

man = {t: {'recipe': {m: w for m, w in zip(f['members'], f['weights'])}, 'threshold': f['thr'],
           'n_checkpoints': sum(15 if 'bag' in m else 5 for m in f['members']),
           'target': 'nucleic acid (DNA or RNA)' if t == 'nuclear' else f'{t.upper()} only',
           'selected_in': {'nuclear': 'notebook 09 (C3_union)', 'dna': 'notebook 12 (S1_dna_pair)',
                           'rna': 'notebook 13 (S1_rna_pair)'}[t]} for t, f in FINALS.items()}
man['note'] = ('every *_bag member = mean of 3 seeds x 5 official folds; a residue is called positive when '
               'probability >= threshold; thresholds are MCC-optimal on the Dev out-of-fold predictions of that '
               'model against its own target')
(OUT/'FINAL_MODELS.json').write_text(json.dumps(man, indent=2))
print('\nwrote per-residue predictions + SUMMARY.csv + FINAL_MODELS.json ->', OUT)

import zipfile
zp = BASE/'EXPORT'/'final_inference_3_models.zip'
zp.parent.mkdir(exist_ok=True)
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for q in sorted(OUT.rglob('*')):
        if q.is_file():
            z.write(q, q.relative_to(BASE))
print('zip:', zp, f'{zp.stat().st_size/1e6:.1f} MB')

## 3. TASK — DNA and RNA as two standalone tasks

**This is the reportable framing.** Four tasks instead of three: metal, small, DNA, RNA. The nuclear
label stops being a target and becomes the raw material the two new targets are cut from.

The ligand class lives only in the label, never in the input, so every model scores all 300 / 46 proteins
and nobody is told anything. bindEmbed21DL and bindNode24 own a single nucleic-acid model, so that model
answers both targets — exactly as their metal model answers the metal task.

In [ ]:
#%% TASK  SPLIT NUCLEAR INTO A DNA TASK AND AN RNA TASK, FOUR MODELS ON EACH
# ============================================================
# TASK. Treat the benchmark as having FOUR tasks, not three: metal, small, DNA, RNA. There is no "nuclear"
#       target any more — the nuclear labels are only the raw material the DNA and RNA targets are cut from.
#           DNA task: y = 1 on DNA-binding residues, 0 on EVERY other residue of EVERY protein in the split
#           RNA task: y = 1 on RNA-binding residues, 0 on every other residue
#       Nothing is filtered. All 300 / 46 proteins are scored. Negatives include metal sites, small-molecule
#       sites, and the sites of the OTHER nucleic acid — exactly as a metal site is a negative in the small task.
#
#       Inference is the same as for metal: each method runs the one model it owns for this kind of ligand.
#       bindEmbed21DL and bindNode24 own a single nucleic-acid model, so that model answers both the DNA task
#       and the RNA task, just as their metal model answers the metal task. Four lines per task:
#           bindEmbed21DL        its nucleic-acid model
#           bindNode24           its nucleic-acid model
#           our_nuclear_model    our general nucleic-acid model
#           our specialist       our DNA model on the DNA task, our RNA model on the RNA task
#
#       Thresholds for the three lines whose Dev predictions we hold are re-derived on Dev OOF FOR THAT TASK.
#       bindEmbed21DL / bindNode24 keep 0.5 (no Dev predictions available for them), so the threshold-free
#       columns — AUPRC and the matched-budget count — carry the comparison.
# Needs the BL cell (KIND, load_baseline, TEST) and the FI cell (FINALS).
# ============================================================
assert 'FINALS' in dir() and 'KIND' in dir(), 'run the BL cell and the FI cell first'


def task_y(split, t):
    """y for the standalone task: the nuclear labels of proteins of class t, zeros everywhere else."""
    out = []
    for r in SPLIT_DF[split].itertuples():
        nuc = np.asarray(getattr(r, LABEL), np.int8)
        k = KIND.get(str(r.prot_id), 'unknown') if nuc.sum() else 'negative'
        out.append(nuc if k in (t, 'both') else np.zeros_like(nuc))
    return np.concatenate(out).astype(int)


YT2 = {(s, t): task_y(s, t) for s in SPLIT_DF for t in ('dna', 'rna')}
for s in TEST:                 # the cut must lose nothing: every old nuclear site lands in DNA, in RNA, or both
    yd, yr = YT2[s, 'dna'], YT2[s, 'rna']
    assert np.array_equal(yd | yr, Y[s]), s
    print(f'{s}: DNA {yd.sum()} + RNA {yr.sum()} - ca hai {(yd & yr).sum()} = {(yd | yr).sum()} '
          f'(= toan bo vi tri axit nucleic cu)  OK')


def lines(t):
    """The four model lines for task t: name -> key into FINALS, or None for a loaded prediction."""
    return {'bindEmbed21DL': None, 'bindNode24': None,
            'our_nuclear_model': 'nuclear', f'our_{t.upper()}_model': t}


def score(src, name, split):
    return np.asarray(FINALS[src][split] if src else load_baseline(name, split), float)


print()
for t in ('dna', 'rna'):
    L = lines(t)
    thr = {n: (float(best_mcc_threshold(YT2['DevSet1010', t], np.asarray(FINALS[src]['oof'], float))[0])
               if src else 0.5) for n, src in L.items()}
    for split in TEST:
        y = YT2[split, t]
        nprot = sum(1 for r in SPLIT_DF[split].itertuples()
                    if np.asarray(getattr(r, LABEL)).sum() and KIND.get(str(r.prot_id)) in (t, 'both'))
        print(f'################ DICH {t.upper()} — {split} ################')
        print(f'  mau: {len(SPLIT_DF[split])} protein | {len(y)} residue | {int(y.sum())} duong '
              f'({100*y.mean():.2f}%) tu {nprot} protein | {int((y == 0).sum())} am')
        rows = []
        for n, src in L.items():
            p = score(src, n, split)
            pred = (p >= thr[n]).astype(int)
            tp = int(((pred == 1) & (y == 1)).sum())
            fp = int(((pred == 1) & (y == 0)).sum())
            rows.append({'model': n, 'thr': round(thr[n], 4),
                         'AUPRC': average_precision_score(y, p), 'AUROC': roc_auc_score(y, p),
                         'goi_duong': tp + fp, 'DUNG': tp, 'SAI': fp, 'BO_SOT': int(y.sum()) - tp,
                         'precision': precision_score(y, pred, zero_division=0),
                         'recall': recall_score(y, pred, zero_division=0),
                         'F1': f1_score(y, pred, zero_division=0), 'MCC': matthews_corrcoef(y, pred)})
        print(pd.DataFrame(rows).sort_values('AUPRC', ascending=False).round(4).to_string(index=False))
        K = int(y.sum())
        mb = pd.DataFrame([{'model': n, f'DUNG_trong_top_{K}': int(y[np.argsort(-score(src, n, split))[:K]].sum())}
                           for n, src in L.items()]).sort_values(f'DUNG_trong_top_{K}', ascending=False)
        mb['ti_le_%'] = (100 * mb[f'DUNG_trong_top_{K}'] / K).round(1)
        print(f'  -- cung ngan sach: moi model du doan dung {K} residue --')
        print('   ' + mb.to_string(index=False).replace('\n', '\n   '))
        print()

print('################ paired protein bootstrap (B = 10000) ################')
boots = []
for t in ('dna', 'rna'):
    L = lines(t)
    ours = f'our_{t.upper()}_model'
    thr = {n: (float(best_mcc_threshold(YT2['DevSet1010', t], np.asarray(FINALS[src]['oof'], float))[0])
               if src else 0.5) for n, src in L.items()}
    for split in TEST:
        y = YT2[split, t]
        for ref in ('bindEmbed21DL', 'bindNode24', 'our_nuclear_model'):
            x = paired_bootstrap(y, score(L[ref], ref, split), score(L[ours], ours, split),
                                 thr[ref], thr[ours], PROT[split])
            x = x[x.metric.eq('AUPRC')].copy()
            x.insert(0, 'split', split)
            x.insert(0, 'comparison', f'{ours} vs {ref}')
            x.insert(0, 'dich', t.upper())
            boots.append(x)
bb = pd.concat(boots, ignore_index=True)
print(bb[['dich', 'split', 'comparison', 'ref', 'new', 'delta', 'delta_CI95_lo', 'delta_CI95_hi',
          'P(new>ref)']].round(4).to_string(index=False))

OUTT = BASE/'results'/'DNA_RNA_AS_TASKS'
OUTT.mkdir(parents=True, exist_ok=True)
bb.to_csv(OUTT/'bootstrap_B10000.csv', index=False)
print('\nsaved ->', OUTT)

## 4. HIT — plain counts

How many residues each system actually gets right, wrong and misses, broken down by ligand class, plus a
matched-budget table. AUPRC hides the fact that the four systems call very different numbers of residues
positive; this does not.

In [ ]:
#%% HIT  HOW MANY RESIDUES EACH SYSTEM ACTUALLY GETS RIGHT, BROKEN DOWN BY LIGAND CLASS
# ============================================================
# HIT. Plain counts, no AUPRC. For every group of proteins: how many nucleic-acid residues exist, how many each
#      system calls positive, how many of those calls are right, wrong, and how many sites are missed.
#
#      per_ligand routes the way the app does — ONE protein at a time, scored by the model for the ligand the user
#      declared, compared against THAT model's own frozen Dev threshold. No cross-protein ranking is involved, so
#      the scale problem that wrecks the pooled AUPRC never arises here.
#          protein declared DNA   -> dna model,  call = score >= thr_dna
#          protein declared RNA   -> rna model,  call = score >= thr_rna
#          protein declared BOTH  -> positive if EITHER model fires at its own threshold
#          class not declared     -> nuclear model, call = score >= thr_nuclear
#      The other three systems use one model everywhere, at their own threshold (baselines: 0.5, as published).
# Needs the BL cell (KIND, load_baseline, TEST) and the FI cell (FINALS).
# ============================================================
assert 'FINALS' in dir() and 'KIND' in dir(), 'run the BL cell and the FI cell first'
OFH = {s: np.r_[0, np.cumsum(SPLIT_DF[s].sequence.str.len().to_numpy())] for s in TEST}
GROUP_ORDER = ['dna', 'rna', 'both', 'unknown', 'none']
GROUP_VI = {'dna': 'protein DNA', 'rna': 'protein RNA', 'both': 'protein ca hai',
            'unknown': 'protein chua ro loai', 'none': 'protein KHONG gan nucleic'}


def groups(split):
    g = np.empty(len(Y[split]), dtype=object)
    for i, r in enumerate(SPLIT_DF[split].itertuples()):
        nuc = np.asarray(getattr(r, LABEL), np.int8)
        g[OFH[split][i]:OFH[split][i + 1]] = KIND.get(str(r.prot_id), 'unknown') if nuc.sum() else 'none'
    return g


G = {s: groups(s) for s in TEST}
THR_N, THR_D, THR_R = FINALS['nuclear']['thr'], FINALS['dna']['thr'], FINALS['rna']['thr']
print(f'thresholds in use: nuclear {THR_N:.6f} | dna {THR_D:.6f} | rna {THR_R:.6f}\n')


def calls(split):
    """Binary call per residue, for each system."""
    d, r, n = (np.asarray(FINALS[t][split], float) for t in ('dna', 'rna', 'nuclear'))
    g = G[split]
    per = np.where(g == 'dna', d >= THR_D,
          np.where(g == 'rna', r >= THR_R,
          np.where(g == 'both', (d >= THR_D) | (r >= THR_R), n >= THR_N)))
    out = {'per_ligand (route)': per.astype(int), 'nuclear_model': (n >= THR_N).astype(int)}
    for b in ('bindNode24', 'bindEmbed21DL'):
        out[b] = (np.asarray(load_baseline(b, split), float) >= 0.5).astype(int)
    return out


for split in TEST:
    y, g = Y[split], G[split]
    present = [k for k in GROUP_ORDER if (g == k).any()]
    nprot = {k: int(sum(1 for i in range(len(SPLIT_DF[split])) if g[OFH[split][i]] == k)) for k in present}
    print(f'################ {split} — {len(SPLIT_DF[split])} protein, {len(y)} residue, '
          f'{int(y.sum())} residue duong ################')
    for name, pred in calls(split).items():
        rows = []
        for k in present:
            m = g == k
            tp = int(((pred == 1) & (y == 1) & m).sum())
            fp = int(((pred == 1) & (y == 0) & m).sum())
            fn = int(((pred == 0) & (y == 1) & m).sum())
            rows.append({'nhom': GROUP_VI[k], 'protein': nprot[k], 'duong_that': tp + fn,
                         'du_doan_duong': tp + fp, 'DUNG': tp, 'SAI': fp, 'BO_SOT': fn,
                         'bat_duoc_%': round(100 * tp / (tp + fn), 1) if tp + fn else None})
        t = pd.DataFrame(rows)
        tot = {'nhom': 'TONG', 'protein': int(t.protein.sum()), 'duong_that': int(t.duong_that.sum()),
               'du_doan_duong': int(t.du_doan_duong.sum()), 'DUNG': int(t.DUNG.sum()),
               'SAI': int(t.SAI.sum()), 'BO_SOT': int(t.BO_SOT.sum()),
               'bat_duoc_%': round(100 * t.DUNG.sum() / t.duong_that.sum(), 1)}
        t = pd.concat([t, pd.DataFrame([tot])], ignore_index=True)
        assert tot['DUNG'] + tot['BO_SOT'] == int(y.sum()), 'TP + FN must be every positive'
        assert tot['DUNG'] + tot['SAI'] == tot['du_doan_duong'], 'TP + FP must be every call'
        p_ = tot['DUNG'] / max(tot['du_doan_duong'], 1)
        print(f"\n--- {name} --- precision {p_:.4f}  recall {tot['DUNG']/tot['duong_that']:.4f}")
        print(t.to_string(index=False))
    print()

# ---- threshold-free view: give every system the same budget inside each group --------------------
print('################ cung ngan sach: trong moi nhom, lay dung K residue diem cao nhat '
      '(K = so duong that cua nhom) ################')
rows = []
for split in TEST:
    y, g = Y[split], G[split]
    d, r, n = (np.asarray(FINALS[t][split], float) for t in ('dna', 'rna', 'nuclear'))
    SC = {'per_ligand (route)': np.where(g == 'dna', d, np.where(g == 'rna', r,
                                         np.where(g == 'both', np.maximum(d, r), n))),
          'nuclear_model': n,
          'bindNode24': np.asarray(load_baseline('bindNode24', split), float),
          'bindEmbed21DL': np.asarray(load_baseline('bindEmbed21DL', split), float)}
    for k in [q for q in GROUP_ORDER if (g == q).any() and y[g == q].sum()]:
        m = g == k
        K = int(y[m].sum())
        for name, sc in SC.items():
            top = np.argsort(-sc[m])[:K]
            rows.append({'split': split, 'nhom': GROUP_VI[k], 'K': K, 'method': name,
                         'DUNG_trong_top_K': int(y[m][top].sum()),
                         'ti_le_%': round(100 * y[m][top].sum() / K, 1)})
bud = pd.DataFrame(rows)
for split in TEST:
    print(f'\n--- {split} ---')
    print(bud[bud.split == split].pivot_table(index=['nhom', 'K'], columns='method',
                                              values='DUNG_trong_top_K').to_string())

OUTH = BASE/'results'/'HIT_COUNTS'
OUTH.mkdir(parents=True, exist_ok=True)
bud.to_csv(OUTH/'matched_budget.csv', index=False)
print('\nsaved ->', OUTH)

## 5. EX — the inference bundle

Packages the three lines for single-protein inference: 95 checkpoints, the frozen thresholds, and a
display-only calibration map. Produces one zip to download.

In the app there is one protein per request, so there is no cross-protein ranking and nothing to re-scale.
The binary call stays on the raw score (`raw >= thr`), so no reported number moves.

In [ ]:
#%% EX  INFERENCE BUNDLE  (run in Colab, then download the zip)
# ============================================================
# EX. Package the three nucleic-acid model lines for single-protein inference.
#
#   In the app there is ONE protein per request, so there is no cross-protein ranking and nothing to re-scale:
#   the user declares the ligand, we run that model, we compare against that model's own frozen Dev threshold.
#   The scale problem that wrecked the pooled AUPRC simply does not arise here.
#
#   THRESHOLDS STAY ON THE RAW SCORE (call = raw >= thr). Every number already reported uses that, so nothing moves.
#   The isotonic map is shipped alongside as DISPLAY ONLY: it turns a raw score into P(this residue binds <ligand>)
#   so the app can show a meaningful percentage. It is fitted on Dev OOF and never touches the binary call.
# Needs the BL cell (KIND) and the FI cell (FINALS, MEMBERS).
# ============================================================
import json, shutil, zipfile
from sklearn.isotonic import IsotonicRegression
assert 'FINALS' in dir() and 'MEMBERS' in dir() and 'KIND' in dir(), 'run the BL cell and the FI cell first'

V1A, MSA = 'NA_PatchHG_Attention', 'NA_PatchHG_Attention_MS'
ARCH = {'V1': V1A, 'MS': MSA}
# The V1_s0 member of the nuclear ensemble predates notebook 09 and lives in its OWN run folder, not under
# NUCLEAR_V3_IMPROVE/states like the seed-bagged members. Every member therefore carries its source explicitly.
NUC_V1_DIR = BASE/'results'/'NUCLEAR_ULTIMATE_V1_DATA_DRIVEN_PATCH'/'patch_states'/V1A
ST = lambda name, arch: V3_ROOT/'states'/name/arch
# task -> (bundle folder, member name in MEMBERS, weight, source dir, arch). Bags split their weight evenly.
SPEC = {
    'nuclear': [('v1', 'V1_s0', 0.2, NUC_V1_DIR, V1A)]
               + [(f'v1_esmif_prott5_s{s}', f'V1+ESMIF_EF+PROTT5_s{s}', 0.4 / 3,
                   ST(f'V1+ESMIF_EF+PROTT5_s{s}', V1A), V1A) for s in (0, 1, 2)]
               + [(f'ms_esmif_prott5_s{s}', f'MS+ESMIF_EF+PROTT5_s{s}', 0.4 / 3,
                   ST(f'MS+ESMIF_EF+PROTT5_s{s}', MSA), MSA) for s in (0, 1, 2)],
    'dna': [(f'v1_s{s}', f'dna_V1_s{s}', 0.50 / 3, ST(f'dna_V1_s{s}', V1A), V1A) for s in (0, 1, 2)]
           + [(f'ms_s{s}', f'dna_MS_s{s}', 0.50 / 3, ST(f'dna_MS_s{s}', MSA), MSA) for s in (0, 1, 2)],
    'rna': [(f'v1_s{s}', f'rna_V1_s{s}', 0.65 / 3, ST(f'rna_V1_s{s}', V1A), V1A) for s in (0, 1, 2)]
           + [(f'ms_s{s}', f'rna_MS_s{s}', 0.35 / 3, ST(f'rna_MS_s{s}', MSA), MSA) for s in (0, 1, 2)],
}
FOLD_FILES = tuple(f'fold{k}{e}' for k in range(1, 6) for e in ('.pt', '_val.npy', '.json'))


def find_states(member, want, arch):
    """The folder holding fold1..5 for this member: the expected one, else a scan of results/ by member name."""
    if (want/'fold1.pt').exists():
        return want
    for hit in sorted((BASE/'results').rglob('fold1.pt')):
        parts = hit.parent.parts
        if member in parts and (arch in parts or hit.parent.name == arch):
            return hit.parent
    return None


def target_labels(split, t):
    """y for one ligand line. nuclear = the original labels; dna/rna = the nuclear labels of proteins of that class."""
    out = []
    for r in SPLIT_DF[split].itertuples():
        nuc = np.asarray(getattr(r, LABEL), np.int8)
        if t == 'nuclear':
            out.append(nuc)
            continue
        k = KIND.get(str(r.prot_id), 'unknown') if nuc.sum() else 'negative'
        out.append(nuc if k in (t, 'both') else np.zeros_like(nuc))
    return np.concatenate(out).astype(int)


# ---- 1. the weights really do reproduce the locked prediction -------------------------------------
print('===== check: SPEC must rebuild FINALS exactly =====')
for t, terms in SPEC.items():
    assert abs(sum(e[2] for e in terms) - 1.0) < 1e-9, f'{t}: weights do not sum to 1'
    for key in ('oof', 'TestSet300', 'TestSetNew46'):
        rebuilt = sum(w * np.asarray(MEMBERS[m][key], float) for _, m, w, _, _ in terms)
        locked = np.asarray(FINALS[t][key], float)
        d = float(np.abs(rebuilt - locked).max())
        assert d < 1e-9, f'{t}/{key}: SPEC does not reproduce FINALS (max |diff| {d:.3e})'
    print(f'  {t:8s} {len(terms)} members, {5*len(terms)} checkpoints, thr {FINALS[t]["thr"]:.16f}  OK')

# ---- 2. display-only calibration, fitted on Dev ---------------------------------------------------
print('\n===== display map: raw score -> P(residue binds this ligand), fitted on Dev OOF =====')
CALIB = {}
for t in SPEC:
    y, p = target_labels('DevSet1010', t), np.asarray(FINALS[t]['oof'], float)
    iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds='clip').fit(p, y)
    xs = np.unique(np.quantile(p, np.linspace(0, 1, 512)))          # compact knots the app can interpolate
    CALIB[t] = {'x': [float(v) for v in xs], 'y': [float(v) for v in iso.predict(xs)]}
    cf = []
    for f, idx in FOLD_IDX.items():                                  # honest quality of the map, cross-fitted
        tr = np.setdiff1d(np.arange(len(y)), idx)
        m = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds='clip').fit(p[tr], y[tr])
        cf.append(float(np.mean((m.predict(p[idx]) - y[idx]) ** 2)))
    thr_p = float(np.interp(FINALS[t]['thr'], CALIB[t]['x'], CALIB[t]['y']))
    print(f'  {t:8s} Dev positives {int(y.sum()):5d} ({100*y.mean():.2f}%) | cross-fit Brier {np.mean(cf):.5f} | '
          f'threshold {FINALS[t]["thr"]:.4f} displays as {100*thr_p:.1f}%')

# ---- 3. collect the checkpoints -------------------------------------------------------------------
BUNDLE = BASE/'EXPORT'/'inference_bundle'
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
rows = []
for t, terms in SPEC.items():
    for folder, member, w, want, arch in terms:
        src = find_states(member, want, arch)
        if src is None:
            rows.append({'task': t, 'folder': folder, 'member': member, 'arch': arch,
                         'status': f'MISSING — looked in {want}'})
            continue
        dst = BUNDLE/t/folder
        dst.mkdir(parents=True, exist_ok=True)
        miss = [f for f in FOLD_FILES if not (src/f).exists()]
        for f in FOLD_FILES:
            if (src/f).exists():
                shutil.copy2(src/f, dst/f)
        rows.append({'task': t, 'folder': folder, 'member': member, 'arch': arch, 'weight': round(w, 6),
                     'files': len(FOLD_FILES) - len(miss), 'src': str(src),
                     'status': 'ok' if not miss else f'{len(miss)} missing'})
inv = pd.DataFrame(rows)
print('\n===== checkpoints collected =====')
print(inv.to_string(index=False))
assert (inv.status == 'ok').all(), 'some members are incomplete — do not ship this bundle'

# ---- 4. the spec the application reads -------------------------------------------------------------
spec = {'created': str(pd.Timestamp.now('UTC')), 'splits_trained_on': 'DevSet1010 (official bindNode24 5-fold)',
        'call_rule': 'binary call = raw_score >= threshold; the calibration map is for display only',
        'tasks': {}}
for t, terms in SPEC.items():
    spec['tasks'][t] = {
        'members': [{'folder': f, 'arch': a, 'weight': float(w), 'folds': 5} for f, m, w, _, a in terms],
        'threshold': float(FINALS[t]['thr']), 'threshold_rule': 'dev_oof_mcc_grid199',
        'n_checkpoints': 5 * len(terms),
        'calibration_display_only': CALIB[t],
        'dev_fold_mean_AUPRC': float(np.mean([average_precision_score(target_labels('DevSet1010', t)[i],
                                                                      np.asarray(FINALS[t]['oof'], float)[i])
                                              for i in FOLD_IDX.values()])),
    }
(BUNDLE/'inference_spec.json').write_text(json.dumps(spec, indent=2))

zp = BASE/'EXPORT'/'inference_bundle.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(BUNDLE.rglob('*')):
        if p.is_file():
            z.write(p, p.relative_to(BUNDLE.parent))
print(f'\nzip: {zp}  {zp.stat().st_size/1e6:.1f} MB  '
      f'({sum(5*len(v) for v in SPEC.values())} checkpoints, 3 ligand lines)')
print('download this one file; it carries the weights, the weights-per-member, the thresholds and the display maps.')